## healpix pixelの3次元座標をvertex shaderで計算する
* 動機
  * healpix pixelは平面ではないので角の4点だけでなく内部にアンカーポイントが必要（特に極付近のピクセル）。
  * 極付近で十分アンカーポイントを取ろうとすると１辺に16個ほど必要
  * 即時にそれだけの3次元座標を計算してVAOにするのは無理
* 案
  * ~~vertex shaderでsphirical coordから計算~~
    * vertex shaderのsin, cosの精度が低くて無理
  * vertex shaderで3次元座標を[bicubic補完](https://en.wikipedia.org/wiki/Bicubic_interpolation)して計算
    * これを実装する

### vertex shaderで3次元座標をbicubic補完して計算
頂点の座標
$$
\vec v = \vec v(p, q) = (v_x, v_y, v_z)
$$
のbicubic補完を考える。

$\vec v(p, q)$のbicubic補完を$\vec u(p, q)$とする。
$$
\vec u = \vec u(p, q) = (u_x, u_y, u_z)
$$
bicubic補完の係数を
$$
u_x(p, q) = \sum_{i=0}^3 \sum_{j=0}^3 \alpha_{xij} p^i q^j \\
u_y(p, q) = \sum_{i=0}^3 \sum_{j=0}^3 \alpha_{yij} p^i q^j \\
u_z(p, q) = \sum_{i=0}^3 \sum_{j=0}^3 \alpha_{zij} p^i q^j \\
$$
とおく。

* $p, q$: attribute float
  * ピクセルの辺に沿って$p \in [0, 1], q \in [0, q]$の範囲で変化
* $\alpha_{\{x,y,z\}}$: uniform mat4
  * bicubic補完の係数

この係数$\alpha$を求める。

まずは$x$成分に注目する。簡単のために$v = v_x, u = u_x, \alpha = \alpha_x$と書く。

ここで$f_{klmn}$を次のように定義する。
$$
f_{klmn} = \left(\frac \partial {\partial p}\right)^{k} 
\left(\frac \partial {\partial q}\right)^{l}
v(m, n) \\
k = \{0, 1\}, l = \{0, 1\}, m = \{0, 1\}, n = \{0, 1\}
$$
この16個の値と$\alpha$について次の関係がある。
$$
\begin{bmatrix}
a_{00}&a_{01}&a_{02}&a_{03}\\a_{10}&a_{11}&a_{12}&a_{13}\\a_{20}&a_{21}&a_{22}&a_{23}\\a_{30}&a_{31}&a_{32}&a_{33}
\end{bmatrix}
=
\begin{bmatrix}
1&0&0&0\\0&0&1&0\\-3&3&-2&-1\\2&-2&1&1
\end{bmatrix}
\begin{bmatrix}
f_{0000}&f_{0001}&f_{0100}&f_{0101}\\f_{0010}&f_{0011}&f_{0110}&f_{0111}\\f_{1000}&f_{1001}&f_{1100}&f_{1101}\\f_{1010}&f_{1011}&f_{1110}&f_{1111}
\end{bmatrix}
\begin{bmatrix}
1&0&-3&2\\0&0&3&-2\\0&1&-2&1\\0&0&-1&1
\end{bmatrix}
$$
$f_{klmn}$を計算し$\alpha$を求める。

$\vec v$のxyz表示は次のようになる。
```
void tu2za(in float t, in float u, out float z, out float a) {
    float abs_u = abs(u);
    if (abs_u <= PI / 4.) { // equatorial belt
        z = 8. / (3. * PI) * u;
        a = t;
    }
    else { // polar caps
        float t_t = mod(t, PI / 2.);
        a = t - (abs_u - PI / 4.) / (abs_u - PI / 2.) * (t_t - PI / 4.);
        float tmp1 = 2. - 4. * abs_u / PI;
        z = sign(u) * (1. - 1. / 3. * tmp1 * tmp1);
    }
}


vec3 position(void) {
    float z, a;
    float p = a_coord[0];
    float q = a_coord[1]; 
    float t0 = u_tu0[0];
    float u0 = u_tu0[1];
    float d = u_pixSize;
    tu2za(t0 + d * (1. - p - q), u0 + d * (p - q), z, a);
    float r = sqrt(1. - z*z);
    return vec3(r * cos(a), r * sin(a), z);
}
```

手計算は大変なのでsympyを使う

In [1]:
from sympy import *

#mysimp = simplify
mysimp = lambda x: x

init_printing()

var('t0 u0 d quadrant ns')

# quadrant: [t/pi * 2]
# ns: north ? 1 : -1

def tu2za_equatorial(t, u):
    z = 8 / (3*pi) * u
    a = t
    return mysimp(z), mysimp(a)

def tu2za_polar_cap(t, u):
    '''
        const t_t = t % (Math.PI / 2)
        const a = t - (abs_u - PI_4) / (abs_u - PI_2) * (t_t - PI_4)
        const z = sign(u) * (1 - 1 / 3 * square(2 - 4 * abs_u / PI))
        return { z, a }
    '''
    t_t = t - quadrant * (pi/2)
    abs_u = ns * u
    # a = t - (abs_u - pi/4) / (abs_u - pi/2) * (t_t - pi/4)
    # a = t - 0.1* (abs_u - pi/4)/(abs_u - pi/2) * (t_t - pi/4)
    a = t + 0.5* (t_t - pi/4)    
    z = ns * (1 - Rational(1,3) * (2 - 4*abs_u/pi )**2)
    return mysimp(z), mysimp(a)
    
def za2xyz(z, a):
    r = sqrt(1 - z**2)
    x = r * cos(a)
    y = r * sin(a) 
    return mysimp(x), mysimp(y), mysimp(z)

def tu2xyz(t, u):
    z, a = tu2za(t, u)
    x, y, z = za2xyz(z, a)
    return x, y, z

def pq2tu(p, q):
    t = t0 + d * (1 - p - q)
    u = u0 + d * (p - q)
    return t, u

def pq2xyz(p, q, tu2za):
    t, u = pq2tu(p, q)
    z, a = tu2za(t, u)
    x, y, z = za2xyz(z, a)
    return x, y, z

In [2]:
import itertools

klmn = [
    (0, 0, 0, 0), (0, 0, 1, 0), (1, 0, 0, 0), (1, 0, 1, 0),
    (0, 0, 0, 1), (0, 0, 1, 1), (1, 0, 0, 1), (1, 0, 1, 1),
    (0, 1, 0, 0), (0, 1, 1, 0), (1, 1, 0, 0), (1, 1, 1, 0),
    (0, 1, 0, 1), (0, 1, 1, 1), (1, 1, 0, 1), (1, 1, 1, 1),]

def formula(tu2za, axis):
    p, q = symbols('p q')
    v = pq2xyz(p, q, tu2za)
    
    def b(k, l, m, n):
        f = diff(diff(v[axis], q, l), p, k)
        f = f.subs({'p': m, 'q': n})
        f = mysimp(f)
        return f
    
    return list(itertools.starmap(b, klmn))
    
ring = {
    'equatorial': tu2za_equatorial,
    'polar_cap': tu2za_polar_cap,
}

In [3]:
b = dict((name, [simplify(formula(tu2za, axis)) for axis in range(3)]) for name, tu2za in ring.items())

次のコードでjs用の計算式が得られる

In [4]:
def js_f_array(ring):
    raw = [[jscode(v) for v in b[ring][axis]] for axis in range(3)]
    js = '[' + ',\n'.join(['[' + ",\n".join(raw[axis]) + ']' for axis in range(3)]) + ']'
    return js

In [5]:
import subprocess

def pbcopy(text):
    p = subprocess.Popen(['pbcopy'], stdin=subprocess.PIPE)
    p.stdin.write(text.encode('ascii'))
    p.stdin.close()
    p.wait()

In [6]:
pbcopy(js_f_array('polar_cap'))

ここまでで$\alpha$を求めることができた。
次の関係を使うと
$$
u(x,y) =
\begin{bmatrix}1&p&p^2&p^3\end{bmatrix}
\begin{bmatrix}
a_{00}&a_{01}&a_{02}&a_{03}\\a_{10}&a_{11}&a_{12}&a_{13}\\a_{20}&a_{21}&a_{22}&a_{23}\\a_{30}&a_{31}&a_{32}&a_{33}
\end{bmatrix}
\begin{bmatrix}1\\q\\q^2\\q^3\end{bmatrix}
$$
vertex shaderでは
```
vec4 p4 = vec4(1., p, p*p, p*p*p);
vec4 q4 = vec4(1., q, q*q, q*q*q);
vec3 position = vec3(
    dot(p4, u_ax * q4),
    dot(p4, u_ay * q4),
    dot(p4, u_az * q4),
);
```
と書ける。```u_a{x,y,z}```は$\alpha_{\{x,y,a\}}$のこと。

### 別のノート

### $f(x,y)$を $p(x,y)=\sum_{i,j} \alpha_{ij} x^i y^j$で近似する方法

$z_k = f(x_k, y_k)$の組から$\alpha_{ij}$を決める。

In [1]:
from sympy import *

init_printing()

In [2]:
n = 4

In [3]:
def index2xy(index):
    i = index // n
    j = index % n
    mesh = [Rational(k, (n-1)) for k in range(n+1)]
#     mesh = [0, Rational(1,4), Rational(1,2), Rational(3,4), 1]
#     mesh = [0, Rational(1,4), Rational(2,4), Rational(3,4), 1]
#     mesh = [-1, -Rational(1,3), Rational(1,3), 1]
    return mesh[i], mesh[j]

In [4]:
def B():
    rows = []
    for index in range(n**2):
        x, y = index2xy(index)
        row = [x**(n-j-1) * y**(n-i-1) for i in range(n) for j in range(n)]
        rows.append(row)
    return Matrix(rows)

In [5]:
b = B()

In [6]:
ib = b**-1

In [7]:
for i in range(n**2):
    print(','.join(jscode(c) for c in ib.transpose()[n**2*i : n**2*(i+1)]) + ',')

81/4,-81/2,99/4,-9/2,-81/2,81,-99/2,9,99/4,-99/2,121/4,-11/2,-9/2,9,-11/2,1,
-243/4,243/2,-297/4,27/2,405/4,-405/2,495/4,-45/2,-81/2,81,-99/2,9,0,0,0,0,
243/4,-243/2,297/4,-27/2,-81,162,-99,18,81/4,-81/2,99/4,-9/2,0,0,0,0,
-81/4,81/2,-99/4,9/2,81/4,-81/2,99/4,-9/2,-9/2,9,-11/2,1,0,0,0,0,
-243/4,405/4,-81/2,0,243/2,-405/2,81,0,-297/4,495/4,-99/2,0,27/2,-45/2,9,0,
729/4,-1215/4,243/2,0,-1215/4,2025/4,-405/2,0,243/2,-405/2,81,0,0,0,0,0,
-729/4,1215/4,-243/2,0,243,-405,162,0,-243/4,405/4,-81/2,0,0,0,0,0,
243/4,-405/4,81/2,0,-243/4,405/4,-81/2,0,27/2,-45/2,9,0,0,0,0,0,
243/4,-81,81/4,0,-243/2,162,-81/2,0,297/4,-99,99/4,0,-27/2,18,-9/2,0,
-729/4,243,-243/4,0,1215/4,-405,405/4,0,-243/2,162,-81/2,0,0,0,0,0,
729/4,-243,243/4,0,-243,324,-81,0,243/4,-81,81/4,0,0,0,0,0,
-243/4,81,-81/4,0,243/4,-81,81/4,0,-27/2,18,-9/2,0,0,0,0,0,
-81/4,81/4,-9/2,0,81/2,-81/2,9,0,-99/4,99/4,-11/2,0,9/2,-9/2,1,0,
243/4,-243/4,27/2,0,-405/4,405/4,-45/2,0,81/2,-81/2,9,0,0,0,0,0,
-243/4,243/4,-27/2,0,81,-81,18,0,-81/4,8

### 移動量マップを作る
* 極ピクセルのみ。
* texture2Dで移動量を変換.